# Compare cell types across multiome datasets

Start with a row per dataset, sample or library, and harmonized cell type. Compare the same cell type across datasets where it exists. Use the per-cell tables to see whether a group difference reflects a shift across cells or a few outliers.

The QC section uses the RNA and ATAC count layers and pairs cells by observation name. Set the file paths below or run the cells in a notebook where `mdata` already exists. Run datasets separately if they use different peak sets. Within a file, the matrix dimensions describe the shared feature universe. Outer-joined matrices require an assay-specific feature mask to distinguish structural missingness from zero counts.

Use a biological sample or donor column for replication if available. The example groups by `library` because it is present in the supplied object. Set `SAMPLE_COL = None` to pool each dataset and cell type. Harmonize cell-type labels upstream and check unlabeled cells before comparing groups.

In [1]:
from pathlib import Path
import pandas as pd

PROJECT_DIR = Path("/gpfs/Labs/Uzun/SCRIPTS/PROJECTS/2024.SINGLE_CELL_GRN_INFERENCE.MOELLER")
PROJECT_DATA_DIR = Path("/gpfs/Labs/Uzun/DATA/PROJECTS/2024.SINGLE_CELL_GRN_INFERENCE.MOELLER")
RAW_DATA_DIR = PROJECT_DATA_DIR / "RAW_DATA" / "HUMAN"

full_metadata_df = pd.read_csv(PROJECT_DATA_DIR / "RAW_DATA" / "HUMAN" / "metadata_summary.tsv", sep="\t", header=0)

ORGANISM_CODE = "hg38"
DATASET_NAME = "ENCODE_colon"
SAMPLE_NAME = "colon_adult_ENCDO856ZOJ"

sample_metadata_df = full_metadata_df[
    (full_metadata_df["dataset"] == DATASET_NAME) &
    (full_metadata_df["library"] == SAMPLE_NAME)
    ]

assert sample_metadata_df["status"].iloc[0] == "ready", f"Sample is not ready: {sample_metadata_df['status'].iloc[0]}"

information_cols = ["dataset", "library", "tissue", "age", "condition", "sex_reported", "status_note"]
for col in information_cols:
    print(f"{col}: {sample_metadata_df[col].iloc[0]}")

TISSUE = sample_metadata_df["tissue"].iloc[0]
sample_h5mu_path = sample_metadata_df["raw_h5mu"].iloc[0]
sample_raw_data_dir = sample_metadata_df["raw_folder"].iloc[0]

PEAK_DATA_DIR = PROJECT_DATA_DIR / "RAW_DATA" / "HUMAN" / "peaks_macs3" / DATASET_NAME

frag_path = RAW_DATA_DIR / sample_raw_data_dir / "fragments_atac_barcodes.tsv.gz"
input_h5mu = RAW_DATA_DIR / sample_h5mu_path

assert frag_path.exists(), f"Fragment file not found: {frag_path}"
print(f"Fragment file found: {frag_path.name}")

assert input_h5mu.exists(), f"Input h5mu file not found: {input_h5mu}"
print(f"Input h5mu file found: {input_h5mu.name}")

sample_metadata_df


dataset: ENCODE_colon
library: colon_adult_ENCDO856ZOJ
tissue: left colon
age: 59
condition: adult
sex_reported: female
status_note: MACS3 peaks and raw .h5mu built
Fragment file found: fragments_atac_barcodes.tsv.gz
Input h5mu file found: colon_adult_ENCDO856ZOJ_raw.h5mu


,dataset,accession,repository,library,status,status_note,tissue,donor,age,sex_reported,...,hc_median_mito_pct,rna_atac_ari,frag_cells_seen_pct,peak_cells,peaks_final,peak_width_median,frip,raw_folder,peaks_folder,raw_h5mu
5,ENCODE_colon,ENCODE,ENCODE portal,colon_adult_ENCDO856ZOJ,ready,MACS3 peaks and raw .h5mu built,left colon,ENCDO856ZOJ,59,female,...,0.36,0.332,NaN,12955.0,114463.0,630.0,0.4957,ENCODE_colon/colon_adult_ENCDO856ZOJ,peaks_macs3/ENCODE_colon/colon_adult_ENCDO856ZOJ,peaks_macs3/ENCODE_colon/colon_adult_ENCDO856Z...


In [3]:
ready_datasets_df = full_metadata_df[full_metadata_df["status"] == "ready"]
ready_datasets_df.to_csv(RAW_DATA_DIR / "ready_datasets.tsv", sep="\t", index=False)

In [2]:
import anndata as ad
import muon as mu

mdata = mu.read_h5mu(input_h5mu)
print("Loaded MuData object from input h5mu file.")

rna = mdata["rna"]
atac = mdata["atac"]

mdata

2026-11-06 13:57:55.712973: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-11-06 13:57:55.769258: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
/gpfs/Home/esm5360/miniconda3/envs/my_env/lib/python3.10/site-packages/muon/_core/preproc.py:32: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('scanpy')` instead
  if Version(scanpy.__version__) < Version("1.10"):
/gpfs/Home/esm5360/miniconda3/envs/my_env/lib/python3.10/site-packages/mudata/_core/mudata.py:1403: FutureWarning: From 0.4 .updat

Loaded MuData object from input h5mu file.


MuData object with n_obs × n_vars = 12955 × 173989
  obs:	'dataset', 'library'
  uns:	'source'
  2 modalities
    rna:	12955 x 59526
      obs:	'barcode_rna'
      var:	'gene_id', 'gene_ids'
    atac:	12955 x 114463
      var:	'chrom', 'start', 'end'

In [3]:
import json

peak_summary = PEAK_DATA_DIR / SAMPLE_NAME / "peaks_summary.json"
h5mu_summary = PEAK_DATA_DIR / SAMPLE_NAME / "h5mu_summary.json"

with open(peak_summary) as f:
    peak_summary_dict = json.load(f)

with open(h5mu_summary) as f:
    h5mu_summary_dict = json.load(f)

display(peak_summary_dict)
display(h5mu_summary_dict)

{'dataset': 'ENCODE_colon',
 'library': 'colon_adult_ENCDO856ZOJ',
 'cells': 12955,
 'cells_seen_first_10M_fragments': 12955,
 'fragments_counted_by_macs3': 184674233,
 'peaks_macs3': 115767,
 'peaks_on_chr1_22_X_Y': 115603,
 'peaks_final': 114463,
 'width_median': 630,
 'width_mean': 769,
 'width_max': 4599,
 'bp_total': 88040245,
 'method': 'macs3 3.0.4 callpeak -f FRAG --barcodes -g hs -q 0.01; chr1-22,X,Y; ENCODE hg38 blacklist v2 removed'}

{'dataset': 'ENCODE_colon',
 'library': 'colon_adult_ENCDO856ZOJ',
 'cells': 12955,
 'genes': 59526,
 'rna_integer_counts': True,
 'median_rna_umi': 3803.0,
 'peaks': 114463,
 'fragments_in_cells': 184674233,
 'frip': 0.4957,
 'median_atac_counts': 6140.0,
 'file': '/gpfs/Labs/Uzun/DATA/PROJECTS/2024.SINGLE_CELL_GRN_INFERENCE.MOELLER/RAW_DATA/HUMAN/peaks_macs3/ENCODE_colon/colon_adult_ENCDO856ZOJ/colon_adult_ENCDO856ZOJ_raw.h5mu'}

In [ ]:
from pathlib import Path
import muon as mu

PROJECT_DIR = Path("/gpfs/Labs/Uzun/SCRIPTS/PROJECTS/2024.SINGLE_CELL_GRN_INFERENCE.MOELLER")

SAMPLE_NAME = "colon_adult_ENCDO856ZOJ"
ORGANISM_CODE = "hg38"
TISSUE = "colon"
DATASET_NAME = "ENCODE_colon"

PROCESSED_DATA_DIR = PROJECT_DIR / "data" / "processed" / ORGANISM_CODE / DATASET_NAME
SAMPLE_PROCESSED_DATA_DIR = PROCESSED_DATA_DIR / SAMPLE_NAME

mdata = mu.read_h5mu(SAMPLE_PROCESSED_DATA_DIR / "multiome_processed.h5mu")
mdata

/gpfs/Home/esm5360/miniconda3/envs/my_env/lib/python3.10/site-packages/mudata/_core/mudata.py:1403: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("var", axis=0, join_common=join_common)
/gpfs/Home/esm5360/miniconda3/envs/my_env/lib/python3.10/site-packages/mudata/_core/mudata.py:1275: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("obs", axis=1, join_common=join_common)


MuData object with n_obs × n_vars = 7584 × 136471
  obs:	'dataset', 'library'
  var:	'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts'
  uns:	'source'
  2 modalities
    rna:	7584 x 25573
      obs:	'barcode_rna', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_20_genes', 'total_counts_mt', 'log1p_total_counts_mt', 'pct_counts_mt', 'total_counts_ribo', 'log1p_total_counts_ribo', 'pct_counts_ribo', 'outlier', 'mt_outlier', 'doublet_score', 'predicted_doublet', 'leiden_res0_25', 'leiden_res0_5', 'leiden_res1', 'leiden_res1_25', 'Epithelial (general)_score', 'Colonocyte (upper crypt)_score', 'Colonocyte (lower crypt)_score', 'BEST4+ colonocyte_score', 'Endothelial_score', 'Pericyte/smooth muscle_score', 'Fibroblast_score', 'T cell_score', 'NK cell_score', 'B cell_score', 'Plasma cell_score', 'Monocyte_score', 'Macrophage_score', 'Dendritic cell_score', 'Mast cell_score', 'Colonocyte_score', 'Goblet cell_score', 'Intestinal stem/TA cell_score', 'Enteroendocrine cell_score', 'Tuft cell_score', 'Smooth muscle_score', 'Enteric glia/neuron_score', 'Cycling cells_score', 'celltype'
      var:	'gene_id', 'gene_ids', 'mt', 'ribo', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts', 'n_cells', 'highly_variable', 'means', 'dispersions', 'dispersions_norm'
      uns:	'celltype_colors', 'dendrogram_leiden_res1', 'hvg', 'leiden_res0_25', 'leiden_res0_25_colors', 'leiden_res0_5', 'leiden_res0_5_colors', 'leiden_res1', 'leiden_res1_25', 'leiden_res1_25_colors', 'leiden_res1_colors', 'log1p', 'neighbors', 'pca', 'scrublet', 'umap'
      obsm:	'X_pca', 'X_umap'
      varm:	'PCs'
      layers:	'counts'
      obsp:	'connectivities', 'distances'
    atac:	7584 x 110898
      obs:	'n_genes_by_counts', 'total_counts', 'tss_score', 'outlier', 'NS', 'nucleosome_signal', 'n_counts', 'leiden'
      var:	'chrom', 'start', 'end', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'nearest_gene', 'TSS_dist', 'TSS_dist_score'
      uns:	'files', 'leiden', 'leiden_colors', 'log1p', 'lsi', 'neighbors', 'peak_gene_distance_params', 'peak_gene_links', 'umap'
      obsm:	'X_lsi', 'X_umap'
      varm:	'LSI'
      layers:	'counts'
      obsp:	'connectivities', 'distances'

In [ ]:
import numpy as np
import pandas as pd
from scipy import sparse

KEYS = ["dataset", "sample", "celltype"]
MUDATA_PATHS = {}  # For example {"colon": "/path/to/colon.h5mu"}
# Use mdata.obs["dataset"] if None
SAMPLE_COL = "library"
CELLTYPE_COL = "celltype"
MIN_CELLS = 10
MIN_FRACTION = 0.01

# Use a unique RNA identifier with the same namespace as your regulatory tables.
# None uses rna.var_names. Set to "gene_id" if that is the shared identifier.
GENE_ID_COL = None

In [ ]:
def _counts(adata):
    matrix = sparse.csr_matrix(adata.layers["counts"], copy=True)
    matrix.sum_duplicates()
    matrix.eliminate_zeros()
    if not np.isfinite(matrix.data).all() or (matrix.data < 0).any():
        raise ValueError("Count layers must contain finite, nonnegative values.")
    return matrix


def _depth(matrix):
    return np.asarray(matrix.sum(axis=1, dtype=np.float64)).ravel()


def log_cp10k(counts):
    depth = _depth(counts)
    scale = np.divide(1e4, depth, out=np.zeros_like(depth), where=depth > 0)
    normalized = counts.multiply(scale[:, None]).tocsr()
    normalized.data = np.log1p(normalized.data)
    return normalized


def prepare_mudata(mdata, dataset_id=None, sample_col="library",
                   celltype_col="celltype", gene_id_col=None):
    rna, atac = mdata.mod["rna"], mdata.mod["atac"]
    if not all(obj.obs_names.is_unique for obj in (rna, atac, mdata)):
        raise ValueError("Observation names must be unique in MuData and both modalities.")

    shared = rna.obs_names[rna.obs_names.isin(atac.obs_names)]
    if len(shared) == 0:
        raise ValueError("No shared cell IDs. Check the RNA-to-ATAC barcode mapping.")
    global_obs = mdata.obs.reindex(shared)
    rna_obs = rna.obs.reindex(shared)
    atac_obs = atac.obs.reindex(shared)
    meta = pd.DataFrame(index=shared)
    meta["dataset"] = global_obs["dataset"] if dataset_id is None else dataset_id
    meta["sample"] = global_obs[sample_col] if sample_col is not None else "pooled"
    meta["celltype"] = rna_obs[celltype_col]
    if meta[["dataset", "sample"]].isna().any().any():
        raise ValueError("Some paired cells are missing a dataset or sample label.")
    n_unlabeled = int(meta["celltype"].isna().sum())
    if n_unlabeled or len(shared) != rna.n_obs or len(shared) != atac.n_obs:
        print(f"Paired {len(shared):,} cells from RNA {rna.n_obs:,} and ATAC {atac.n_obs:,}. "
              f"Groups exclude {n_unlabeled:,} missing cell-type labels.")

    for prefix, obs, columns in [
        ("rna", rna_obs, ["total_counts", "pct_counts_mt"]),
        ("atac", atac_obs, ["total_counts", "n_counts", "tss_score", "nucleosome_signal"]),
    ]:
        for col in columns:
            if col in obs:
                meta[f"{prefix}_obs_{col}"] = obs[col]

    gene_ids = rna.var_names if gene_id_col is None else rna.var[gene_id_col]
    genes = pd.Index(gene_ids, name="gene")
    peaks = pd.Index(atac.var_names, name="peak")
    if genes.hasnans or peaks.hasnans or not genes.is_unique or not peaks.is_unique:
        raise ValueError("Gene and peak identifiers must be unique and nonmissing.")

    rna_counts = _counts(rna)[rna.obs_names.get_indexer(shared)]
    atac_counts = _counts(atac)[atac.obs_names.get_indexer(shared)]
    return meta, rna_counts, atac_counts, genes, peaks


def _add_quantiles(row, name, values):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    quantiles = np.quantile(values, [0.1, 0.5, 0.9]) if values.size else [np.nan] * 3
    row.update({f"{name}_{suffix}": value
                for suffix, value in zip(("p10", "median", "p90"), quantiles)})


def profile_cells(prepared, min_cells=10, min_fraction=0.01):
    if min_cells < 1 or not 0 < min_fraction <= 1:
        raise ValueError("Use min_cells >= 1 and 0 < min_fraction <= 1.")
    meta, rna, atac, genes, peaks = prepared
    cell_qc = meta.copy()
    cell_qc["rna_counts_in_matrix"] = _depth(rna)
    cell_qc["atac_counts_in_matrix"] = _depth(atac)
    cell_qc["genes_per_cell"] = rna.getnnz(axis=1)
    cell_qc["peaks_per_cell"] = atac.getnnz(axis=1)
    expr = log_cp10k(rna)
    summaries, gene_tables, peak_tables = [], [], []
    for key, rows in cell_qc.groupby(KEYS, observed=True, sort=False).indices.items():
        labels = dict(zip(KEYS, key))
        n = len(rows)
        cutoff = max(min_cells, int(np.ceil(n * min_fraction)))
        group = {**labels, "n_cells": n, "feature_detection_cutoff_cells": cutoff}
        for col in cell_qc.columns.difference(KEYS, sort=False):
            _add_quantiles(group, col, cell_qc[col].iloc[rows])

        for label, counts, ids, tables in [
            ("gene", rna[rows], genes, gene_tables),
            ("peak", atac[rows], peaks, peak_tables),
        ]:
            detected = counts.getnnz(axis=0)
            total = np.asarray(counts.sum(axis=0, dtype=np.float64)).ravel()
            features = pd.DataFrame({
                **labels,
                label: ids,
                "n_cells_detected": detected,
                "frac_cells_detected": detected / n,
                "mean_raw_count": total / n,
                "mean_count_when_detected": np.divide(
                    total, detected, out=np.full(len(ids), np.nan), where=detected > 0
                ),
            })
            if label == "gene":
                features["mean_log1p_cp10k"] = np.asarray(expr[rows].mean(axis=0)).ravel()
            tables.append(features)
            group[f"n_{label}s_in_matrix"] = len(ids)
            group[f"n_{label}s_detected_any"] = int(np.count_nonzero(detected))
            group[f"n_{label}s_detected_at_cutoff"] = (
                int(np.count_nonzero(detected >= cutoff)) if n >= cutoff else np.nan
            )
            _add_quantiles(group, f"cells_per_detected_{label}", detected[detected > 0])
            _add_quantiles(group, f"{label}_nonzero_count", counts.data)
        summaries.append(group)

    if not summaries:
        raise ValueError("No paired cells have a cell-type label.")
    return (
        cell_qc.rename_axis("cell_id").reset_index(),
        pd.DataFrame(summaries),
        pd.concat(gene_tables, ignore_index=True),
        pd.concat(peak_tables, ignore_index=True),
    )

In [ ]:
def summarize(mdata, dataset_id):
    prepared = prepare_mudata(mdata, dataset_id, SAMPLE_COL, CELLTYPE_COL, GENE_ID_COL)
    return profile_cells(prepared, MIN_CELLS, MIN_FRACTION)


if MUDATA_PATHS:
    import mudata as md

    profiles = [summarize(md.read_h5mu(path), dataset) for dataset, path in MUDATA_PATHS.items()]
    cell_qc, group_qc, gene_qc, peak_qc = (
        pd.concat(tables, ignore_index=True) for tables in zip(*profiles)
    )
    del profiles
elif "mdata" in globals():
    cell_qc, group_qc, gene_qc, peak_qc = summarize(mdata, DATASET_NAME)

if "group_qc" in globals():
    display(group_qc[KEYS + [
        "n_cells", "genes_per_cell_median", "peaks_per_cell_median",
        "rna_counts_in_matrix_median", "atac_counts_in_matrix_median",
        "n_genes_detected_at_cutoff", "n_peaks_detected_at_cutoff",
    ]])

## Interpret the QC tables

- `cell_qc` has paired-cell depth, detected genes and accessible peaks per cell, plus available stored QC columns. Depth recomputed from the layers is the sum over retained features. Confirm whether the original ATAC metadata refers to fragments, insertions, reads, or counts in peaks before comparing depths.
- `group_qc` has counts and 10th, 50th and 90th percentiles. A detected entry has a raw count above zero. The cutoff uses both an absolute cell count and a group fraction. Compare breadth across equally sized and similarly sequenced groups as a sensitivity check.
- `gene_qc` and `peak_qc` give the cell count and fraction detected for every feature, including zeros. For abundance distributions compare per-feature mean normalized RNA expression or ATAC detection fractions. Raw nonzero count distributions describe positive observations separately from sparsity.
- For per-TF expression, subset `gene_qc` with your model's TF IDs. Report both detection fraction and mean log1p CP10K, which includes cells with zero counts. Track TFs absent from the feature universe separately.

For a dataset outlier plot, pick a shared cell type and plot these group summaries by dataset, with one point per biological sample. Inspect RNA and ATAC depth, per-cell complexity, TSS enrichment and model-feature coverage before interpreting outliers in the regulatory metrics.

In [ ]:
# Optional: provide TF IDs in the namespace used by gene_qc["gene"].
TF_IDS = None
if TF_IDS is not None and "gene_qc" in globals():
    tf_expression = gene_qc.loc[gene_qc["gene"].isin(TF_IDS)].copy()
    tf_missing_by_dataset = {
        dataset: sorted(set(TF_IDS) - set(table["gene"]))
        for dataset, table in gene_qc.groupby("dataset", observed=True)
    }

# Optional: choose a shared cell type to plot one point per sample/library.
PLOT_CELLTYPE = None
if PLOT_CELLTYPE is not None and "group_qc" in globals():
    import matplotlib.pyplot as plt

    plot_qc = group_qc.loc[group_qc["celltype"].eq(PLOT_CELLTYPE)]
    metrics = {
        "rna_counts_in_matrix_median": "Median RNA counts",
        "atac_counts_in_matrix_median": "Median ATAC counts in peaks",
        "genes_per_cell_median": "Median genes per cell",
        "peaks_per_cell_median": "Median peaks per cell",
    }
    fig, axes = plt.subplots(2, 2, figsize=(10, 6), layout="constrained")
    for ax, (column, title) in zip(axes.flat, metrics.items()):
        for dataset, samples in plot_qc.groupby("dataset", observed=True):
            ax.scatter([str(dataset)] * len(samples), samples[column], alpha=0.75)
        ax.set_ylabel(title)
        ax.tick_params(axis="x", labelrotation=30)
    fig.suptitle(PLOT_CELLTYPE)
    plt.show()

## Regulatory features for a selected group

Use the same TF, TG and peak identifiers and genome assembly used by the model. Supply `tf_tg_pairs` with columns `tf` and `tg`, and `peak_tg_pairs` with columns `peak` and `tg`. Extra columns such as `label`, `distance_bp`, or benchmark IDs are retained. For the TF–TG comparison, include ground-truth positives and evaluation background pairs matched on TF identity and TG expression/detection where possible. Treat unlabeled edges as background when negatives have not been experimentally established.

Calculate correlations within a group using paired cells. The code reports signed Pearson correlations and correlations after regressing out log count totals as a sensitivity check. Use RNA depth for TF–TG and both modalities' depths for peak–TG. Keep the fraction of candidate pairs that have enough detected cells and nonzero variance alongside the correlation distribution. Compare peak–TG links against candidate peaks with similar distance, detection, GC and width. [Signac's LinkPeaks documentation](https://stuartlab.org/signac/reference/linkpeaks) describes a background-based approach for peak–gene testing.

In [ ]:
def pair_correlations(x, y, x_ids, y_ids, pairs, x_col, y_col,
                      log_depths=None, min_detected=10, batch_size=128):
    if x.shape[0] != y.shape[0] or len(x_ids) != x.shape[1] or len(y_ids) != y.shape[1]:
        raise ValueError("Pair correlations require aligned cells and feature IDs.")
    result = pairs.copy()
    xi = x_ids.get_indexer(result[x_col])
    yi = y_ids.get_indexer(result[y_col])
    mapped = np.flatnonzero((xi >= 0) & (yi >= 0))
    n = x.shape[0]
    basis = None
    if log_depths is not None and n >= 3:
        design = np.column_stack([np.ones(n), log_depths])
        u, singular, _ = np.linalg.svd(design, full_matrices=False)
        tolerance = singular.max() * max(design.shape) * np.finfo(float).eps
        basis = u[:, singular > tolerance]
    n_x = np.full(len(result), np.nan)
    n_y = np.full(len(result), np.nan)
    r = np.full(len(result), np.nan)
    adjusted = np.full(len(result), np.nan)

    def pearson(a, b):
        norm_a = np.linalg.norm(a, axis=0)
        norm_b = np.linalg.norm(b, axis=0)
        valid = (norm_a > 1e-10) & (norm_b > 1e-10)
        return np.divide((a * b).sum(axis=0), norm_a * norm_b,
                         out=np.full(len(norm_a), np.nan), where=valid)

    for start in range(0, len(mapped), batch_size):
        p = mapped[start:start + batch_size]
        a = x[:, xi[p]].toarray().astype(float)
        b = y[:, yi[p]].toarray().astype(float)
        n_x[p] = (a > 0).sum(axis=0)
        n_y[p] = (b > 0).sum(axis=0)
        enough = (n_x[p] >= min_detected) & (n_y[p] >= min_detected)
        if n < 3 or not enough.any():
            continue
        a = a[:, enough]
        b = b[:, enough]
        a -= a.mean(axis=0)
        b -= b.mean(axis=0)
        r[p[enough]] = pearson(a, b)
        if basis is not None and n - basis.shape[1] >= 2:
            a -= basis @ (basis.T @ a)
            b -= basis @ (basis.T @ b)
            adjusted[p[enough]] = pearson(a, b)

    result["n_cells"] = n
    result["pair_in_feature_universe"] = (xi >= 0) & (yi >= 0)
    result["n_cells_x_detected"] = n_x
    result["n_cells_y_detected"] = n_y
    result["pearson_r"] = r
    result["depth_adjusted_r"] = adjusted
    return result

In [ ]:
# Set to (dataset, sample, celltype) using the keys in group_qc.
SELECT_GROUP = None
REGULATORY_PATH = None  # Use the open mdata, or set a path for the selected group.

if SELECT_GROUP is not None:
    path = REGULATORY_PATH or MUDATA_PATHS.get(SELECT_GROUP[0])
    if path is not None:
        import mudata as md

        regulatory_mdata = md.read_h5mu(path)
        dataset_id = SELECT_GROUP[0]
    else:
        regulatory_mdata = mdata
        dataset_id = DATASET_NAME
    meta, rna, atac, genes, peaks = prepare_mudata(
        regulatory_mdata, dataset_id, SAMPLE_COL, CELLTYPE_COL, GENE_ID_COL
    )
    matches = meta[KEYS].eq(pd.Series(dict(zip(KEYS, SELECT_GROUP)))).fillna(False).all(axis=1)
    rows = np.flatnonzero(matches.to_numpy())
    if len(rows) == 0:
        raise ValueError(f"Group has no cells: {SELECT_GROUP}")
    rna = rna[rows]
    atac = atac[rows]
    rna_depth = _depth(rna)
    atac_depth = _depth(atac)
    paired_with_counts = (rna_depth > 0) & (atac_depth > 0)
    if not paired_with_counts.any():
        raise ValueError("No cells in the group have positive depth in both modalities.")
    expr = log_cp10k(rna[paired_with_counts]).tocsc()
    accessible = atac[paired_with_counts].astype(bool).tocsc()
    log_rna_depth = np.log1p(rna_depth[paired_with_counts])
    log_both_depths = np.column_stack([
        log_rna_depth, np.log1p(atac_depth[paired_with_counts])
    ])

    if "tf_tg_pairs" in globals():
        tf_tg_stats = pair_correlations(
            expr, expr, genes, genes, tf_tg_pairs, "tf", "tg", log_rna_depth
        )
    if "peak_tg_pairs" in globals():
        peak_tg_stats = pair_correlations(
            accessible, expr, peaks, genes, peak_tg_pairs, "peak", "tg", log_both_depths
        )

## Accessible peaks near target genes

For each cell and TG, sum binary accessibility across the peaks assigned to that gene within 100 kb of its TSS. Compare the sum to the TG's normalized RNA across the paired cells. Use the unweighted sum first and optionally repeat with exponential decay. A gene with many candidate peaks has more chances to score highly. Keep the number of candidate peaks and compare across genes with similar candidate counts and distance distributions.

The example uses the closest gene assignment already present in `atac.var`. Confirm the distance unit is bp and the gene IDs match the RNA identifier before running it. To use all TSSs within the window, replace the nearest-link table with a long peak–gene candidate table from the same reference. Use the same rule across datasets. For locus-level comparisons, quantify accessibility in a common peak set within each genome build.

In [ ]:
def local_accessibility(accessible, peaks, genes, links, window_bp=100_000, decay_bp=None):
    if window_bp < 0 or (decay_bp is not None and decay_bp <= 0):
        raise ValueError("Use a nonnegative window and a positive decay scale.")
    links = links[["peak", "tg", "distance_bp"]].copy()
    links["distance_bp"] = pd.to_numeric(links["distance_bp"], errors="raise").abs()
    links = links.loc[
        links["distance_bp"].le(window_bp) & links["peak"].isin(peaks) & links["tg"].isin(genes)
    ]
    links = links.sort_values("distance_bp").drop_duplicates(["peak", "tg"])
    target_ids = pd.Index(links["tg"].unique(), name="tg")
    pi = peaks.get_indexer(links["peak"])
    gi = target_ids.get_indexer(links["tg"])
    weights = (np.ones(len(links)) if decay_bp is None
               else np.exp(-links["distance_bp"].to_numpy() / decay_bp))
    mapping = sparse.csr_matrix((weights, (pi, gi)), shape=(len(peaks), len(target_ids)))
    candidates = np.bincount(gi, minlength=len(target_ids))
    return (accessible @ mapping).tocsc(), target_ids, candidates


RUN_NEAREST = False  # Enable after checking gene IDs and the distance unit.
if SELECT_GROUP is not None and RUN_NEAREST:
    peak_meta = regulatory_mdata.mod["atac"].var.reindex(peaks)
    nearest_links = pd.DataFrame({
        "peak": peaks,
        "tg": peak_meta["nearest_gene"].to_numpy(),
        "distance_bp": peak_meta["TSS_dist"].to_numpy(),
    })
    # Optionally restrict to your model TGs to reduce memory use.
    score, tg_ids, n_candidates = local_accessibility(accessible, peaks, genes, nearest_links)
    if len(tg_ids) == 0:
        raise ValueError("No nearest-TSS links matched the selected gene IDs and distance window.")
    local_pairs = pd.DataFrame({"local_accessibility": tg_ids, "tg": tg_ids,
                                "n_candidate_peaks": n_candidates})
    local_tg_stats = pair_correlations(
        score, expr, tg_ids, genes, local_pairs, "local_accessibility", "tg", log_both_depths
    )
    # For the weighted version, repeat with decay_bp=50_000 as an exploratory setting.

## Motif-bearing peaks and peak sequence composition

Supply `motif_hits` with columns `peak` and `tf`, using your existing peak–motif scan and mapping motifs to TFs. Count unique peaks per TF, and report the fraction detected at the group cutoff. The mean peak detection fraction across all motif-bearing peaks is also useful for small groups.

For sequence composition, optionally supply `peak_sequences` as a pandas Series of reference sequences indexed by peak ID. Extract sequences using the dataset's genome build and verified coordinate convention. The code reports pooled A/C/G/T frequencies over the selected peaks and excludes ambiguous bases from the denominator. Compare the whole peak universe, peaks detected at the group cutoff, and the detected subset of peaks bearing each motif. If peak sets vary across datasets, differences can reflect peak definitions or sequence length.

For motif accessibility enrichment across cells, chromVAR estimates deviations using background peaks matched for GC and average accessibility. Related TFs can share a motif, so interpret those scores at the motif or motif-family level. [chromVAR vignette](https://bioconductor.org/packages/release/bioc/vignettes/chromVAR/inst/doc/Introduction.html)

In [ ]:
def motif_peak_summary(peak_profile, motif_hits, cutoff_cells, n_cells):
    hits = motif_hits[["peak", "tf"]].dropna().drop_duplicates()
    hits = hits.merge(
        peak_profile[["peak", "n_cells_detected", "frac_cells_detected"]],
        on="peak", validate="many_to_one"
    )
    hits["detected_at_cutoff"] = hits["n_cells_detected"].ge(cutoff_cells)
    if n_cells < cutoff_cells:
        hits["detected_at_cutoff"] = np.nan
    return hits.groupby("tf", observed=True).agg(
        n_motif_peaks=("peak", "size"),
        n_motif_peaks_detected=("detected_at_cutoff", lambda s: s.sum(min_count=1)),
        frac_motif_peaks_detected=("detected_at_cutoff", "mean"),
        mean_motif_peak_detection=("frac_cells_detected", "mean"),
    ).reset_index()


def base_frequencies(sequences):
    sequences = sequences.dropna().str.upper()
    totals = pd.Series({base: sequences.str.count(base).sum() for base in "ACGT"})
    total_bases = totals.sum()
    fractions = (totals / total_bases).add_prefix("frac_")
    return pd.Series({
        "n_peaks_with_sequence": len(sequences),
        "n_acgt_bases": total_bases,
        "frac_GC": fractions["frac_G"] + fractions["frac_C"],
        **fractions.to_dict(),
    })


if SELECT_GROUP is not None and "peak_qc" in globals():
    keys = pd.Series(dict(zip(KEYS, SELECT_GROUP)))
    peak_profile = peak_qc.loc[peak_qc[KEYS].eq(keys).all(axis=1)]
    group_profile = group_qc.loc[group_qc[KEYS].eq(keys).all(axis=1)].iloc[0]
    cutoff = int(group_profile["feature_detection_cutoff_cells"])
    if "motif_hits" in globals():
        motif_stats = motif_peak_summary(peak_profile, motif_hits, cutoff, int(group_profile["n_cells"]))
    if "peak_sequences" in globals():
        detected_peaks = peak_profile.loc[peak_profile["n_cells_detected"].ge(cutoff), "peak"]
        whole_peak_composition = base_frequencies(peak_sequences.reindex(peak_profile["peak"]))
        detected_peak_composition = base_frequencies(peak_sequences.reindex(detected_peaks))
        if "motif_hits" in globals():
            detected_hits = motif_hits.loc[motif_hits["peak"].isin(detected_peaks)].dropna(
                subset=["tf", "peak"]
            )
            motif_composition = pd.DataFrame({
                tf: base_frequencies(peak_sequences.reindex(hits["peak"].unique()))
                for tf, hits in detected_hits.groupby("tf", observed=True)
            }).T.rename_axis("tf").reset_index()

## Relate dataset differences to model performance

1. Start with paired cell count, median depth and per-cell complexity, then add TF/TG and ground-truth edge coverage. For each benchmark group, measure positive coverage against the original eligible ground truth and positive prevalence among scored pairs. Record the model's actual cell, TF, TG and peak filters alongside the full-data QC.
2. Compare performance within shared cell types and the same evaluation protocol. For AUPRC, keep evaluated positive prevalence visible and use a fixed edge universe where possible. If benchmark sets differ, use within-benchmark comparisons or report them separately.
3. Repeat comparisons after matching or downsampling cell counts. Where read or fragment depth can be made comparable, run a depth sensitivity analysis as well. Recompute the features and reevaluate the model on those data. Treat raw-count thinning as a sensitivity check with units tied to each assay.
4. Add regulatory metrics next. Useful group summaries include median signed TF–TG correlation for ground-truth and matched background pairs, fraction of pairs with estimable correlation, median depth-adjusted local-accessibility correlation, and TF motif accessibility. Keep per-TF summaries so a change in TF composition does not dominate the pooled metric.
5. Join QC and model scores at the same grain. If scores are pooled by dataset and cell type, rerun QC with `SAMPLE_COL = None` or aggregate at that grain explicitly. For biological inference, use independent samples and account for repeated cell types or methods within the same sample. With only a few datasets, use plots and report associations descriptively. [Squair et al., 2021](https://www.nature.com/articles/s41467-021-25960-2)

In [ ]:
# Optional: model_scores has dataset/sample/celltype and evaluation metric columns.
# Repeated score rows for different methods or benchmarks can share one QC row.
if "model_scores" in globals() and "group_qc" in globals():
    score_qc = model_scores.merge(
        group_qc, on=KEYS, how="left", validate="many_to_one", indicator=True
    )
    missing_qc = score_qc.loc[score_qc["_merge"].eq("left_only"), KEYS].drop_duplicates()
    if not missing_qc.empty:
        display(missing_qc)